# Dr. Case Notebook
# Machine Learning: Linear Regression

REQ: Open notebooks with a standard header including a good title, your company/name/alias, a link to the repo, purpose, and date.

REQ: Include dataset, description, and source information.

- Author: [Denise Case](https://github.com/denisecase/)
- Repository: [datafun-07-regression](https://github.com/denisecase/datafun-07-regression/)
- Purpose: Demonstrate a repeatable workflow for a new, unexplored dataset
- Date: 2026-06

## Instructions

- Scan the headings first, to get an idea of the process and overall goal. 
- The provided content is a well-structured example that shows the process.
- Seek to understand the process and presentation of the work. 
- The goal is to be able to perform a similar analysis **on any data you choose**. 
- Clean up your copy of this notebook as you work - remove lines indicating the cell type, etc.

## Dataset Information

- Dataset: Our World in Data CO2 / energy dataset
- A subset with United States, China, India, Germany, United Kingdom,
  France, Japan, Brazil, Canada, World for years 1990-present
  has been **used for the example**.
- GitHub: <https://github.com/owid/co2-data>
- Explorer: <https://ourworldindata.org/explorers/co2>
  
This is a Markdown cell.

## Section 0a. Intro to Jupyter Notebooks

This is a Markdown cell (not a Python cell). 
Here are a few tips to help you get started with Jupyter Notebooks:

- To run a cell, press **Ctrl+Enter** (or **Cmd+Enter** on Mac) when you're done editing the cell.
- You can change the type of a cell (e.g., code or markdown) by looking in the lower left corner of the notebook interface.
- You can rearrange cells by dragging and dropping them within the notebook.
- After creating a new notebook, use **File > Save as** to rename and save it into your project repository `notebooks` folder.
- To select a kernel (Python environment) for your notebook in Visual Studio Code, click on the **Select Kernel** name in the top-right corner of the notebook interface and choose the desired kernel from the dropdown menu. 
- Follow suggestions to install recommended extensions. 
- Once installed, click Select Kernel / Python Environments and choose the Recommended `.venv` option created earlier. 
- This will create a new kernel for the notebook and allow the notebook to use packages installed in the .venv/ environment.

Once skilled with Notebooks, you can delete this entire cell or create custom notes.

This is a Markdown cell.  


## Section 0b. Intro to Linear Regression

### WHEN to test for a linear relationship

Test for linear relationships after EDA, once you have a numeric pair worth pursuing:

- When EDA shows two numeric variables that appear to move together
- When you want to estimate or predict one numeric variable (the target) from another (the feature)
- When you want a simple, interpretable baseline before reaching for a more complex model
- Even when you are not sure the relationship is linear, testing is how you find out

### WHY test for a linear relationship

A linear model is the simplest place to start, and testing it is informative either way:

- Linear regression is the simplest predictive model, so it makes a clear baseline
- It is interpretable: the slope and intercept have direct, reportable meaning
- You do not assume the relationship is linear, you check it with the residual plot, R-squared, and RMSE
- A straight line that does NOT fit is itself a useful finding: it points you to transform a variable, choose a different feature, or use a different model
- Knowing whether a simple line is enough keeps you from over-engineering the analysis

This work tends to follow EDA: EDA finds a relationship worth examining, and regression tests whether a straight line is a fair description of it.

This workflow works for most numeric, **tabular** datasets.

Once you understand linear regression, you can delete this entire cell or create custom notes.

This is a Markdown cell.

## Section 1. Project Setup and Imports

All imports and configuration appear once, at the top of the notebook.

WHY:
- Keeps notebooks readable and reproducible
- Mirrors professional scripts
- Makes it clear what must be installed

This is a Markdown cell.

In [ ]:
# This is a Python cell.


# === Section 1a. DECLARE IMPORTS (BRING IN FREE CODE) ===

import logging  # for type hinting only
from typing import Final  # for type hinting

from datafun_toolkit.logger import get_logger, log_header
from matplotlib.axes import Axes
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression

# Type hint for Axes object (basic plot type returned by Seaborn)
# A seaborn plot is a set of axes. Set title, labels, etc. on the axes.
# A figure can contain multiple axes (plots)
# from matplotlib.figure import Figure

# === Section 1b. CONFIGURE LOGGER ONCE PER NOTEBOOK ===

LOG: logging.Logger = get_logger("P07-NB", level="DEBUG")
log_header(LOG, "P07-NB")

# === Section 1c. Global Constants and Configuration ===

# CUSTOM: These are dataset-specific constants
# used in multiple places in the code.
# Inspect or explore the dataset to determine columns needed for analysis.

# CUSTOM: Data set name
DATASET_NAME: Final[str] = "owid-co2-data-subset"

# ==========================================================
# ANALYST CHOICE:
# Linear regression models one numeric TARGET (y) as a straight-line
# function of one numeric FEATURE (x):  y = slope * x + intercept
#
# Choose the pair from what you saw during EDA. In the EDA script, the
# correlation matrix and the scatter plot examined gdp (x) vs co2 (y),
# so that is the pair we investigate here.
#
# Choosing a pair does NOT mean we believe the relationship is linear.
# We fit the line so we can look at how well (or how poorly) it describes
# the data. That examination is the whole point.
# ==========================================================

# CUSTOM: Grouping column (chose one categorical/non-numeric variable)

# CUSTOM: Grouping column (chose one categorical/non-numeric variable)

GROUP_COL: Final[str] = "country"

# CUSTOM: Numeric columns to analyze (chose 4-5 numeric variables)

SELECTED_NUMERIC_COLS: Final[list[str]] = [
    "year",
    "co2",
    "co2_per_capita",
    "population",
    "gdp",
]

# CUSTOM: One numeric feature (the predictor, plotted on the x-axis)
FEATURE_COL: Final[str] = "gdp"

# CUSTOM: One numeric target (the response, plotted on the y-axis)
TARGET_COL: Final[str] = "co2"

# CUSTOM: Assign readable labels for the charted variables.
FEATURE_LABEL: Final[str] = "GDP"
TARGET_LABEL: Final[str] = "CO2 emissions"

# CUSTOM: A single feature value to predict the target for, as an example.
# Pick a value inside (or near) the range of the data you observed in EDA.
EXAMPLE_FEATURE_VALUE: Final[float] = 1.0e12  # example GDP value


# === Section 1d. Pandas Configuration for Display ===

# Pandas display configuration (helps in notebooks)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

## Section 2. Load the Data

This is a Markdown cell.

WHY: Before analysis, load the data to confirm:

- The dataset loads successfully
- The structure matches expectations
- Column names are available and readable

This is a Markdown cell.

In [ ]:
# Python cell

# Load a dataset into a DataFrame.

# Seaborn provides clean built-in datasets for practice.
# Other projects may load from CSV, JSON, or a database.

# Load the penguins dataset from Seaborn
# Into a pandas DataFrame (2D table)
LOG.info(f"Loading dataset: {DATASET_NAME}")
df: pd.DataFrame = pd.read_csv(f"data/raw/{DATASET_NAME}.csv")
count_of_rows: int = df.shape[0]
count_of_columns: int = df.shape[1]
LOG.info(f"Loaded: {count_of_rows} rows, {count_of_columns} columns")

## Section 3. Prepare a Modeling View (Feature + Target Only)

Create a cleaned view containing only the rows we can model.

Strategy:
- Keep the original DataFrame unchanged
- Drop rows missing the feature OR the target

WHY: A regression cannot use a row that is missing either 
the x value or the y value. 

We remove those rows up front so the model sees only
complete (x, y) pairs.

This is a Markdown cell.

In [ ]:
# Section 3 Python cell

LOG.info("Creating modeling view (dropping rows missing feature or target)")

# The two columns we require to be non-missing.
# FEATURE_COL and TARGET_COL are single strings, so wrap them in a list.
cols_required: list[str] = [FEATURE_COL, TARGET_COL]
LOG.debug(f"Columns required to be non-missing: {cols_required}")

# dropna(subset=...) only looks at the specified columns, not the whole row.
# .copy() creates a new DataFrame so we don't accidentally modify the original.
df_model: pd.DataFrame = df.dropna(subset=cols_required).copy()

In [ ]:
# Section 3 Python cell

# Report what was kept and what was dropped
count_original: int = df.shape[0]
count_model: int = df_model.shape[0]
count_dropped: int = count_original - count_model

LOG.info(f"Original rows: {count_original}")
LOG.info(f"Model rows:    {count_model}")
LOG.info(f"Rows dropped:  {count_dropped}")

## Section 4. Build the Feature Matrix X and Target Vector y

WHY: scikit-learn expects two different shapes:

- X must be 2-D, with shape (n_rows, n_features).
  Even with one feature, it must be (n_rows, 1), that is,
  a column, not a flat list.

- y is 1-D, with shape (n_rows,).

The double-bracket df[[FEATURE_COL]] returns a DataFrame (2-D).
The single-bracket df[TARGET_COL] returns a Series (1-D).
Converting each to a NumPy array gives the shapes sklearn wants.

This is a Markdown cell.


In [ ]:
# Section 4 Python cell

LOG.info("Building feature matrix X and target vector y")

# Double brackets -> DataFrame -> 2-D array of shape (n, 1)
X: np.ndarray = df_model[[FEATURE_COL]].to_numpy()

# Single brackets -> Series -> 1-D array of shape (n,)
y: np.ndarray = df_model[TARGET_COL].to_numpy()

In [ ]:
# Section 4 Python cell

LOG.debug(f"X shape: {X.shape}  (rows, features)")
LOG.debug(f"y shape: {y.shape}  (rows,)")

## Section 5. Fit a Straight Line

WHY: LinearRegression() is the standard tool.
The pattern we follow is create a model,
call .fit() to train it, then read its results
and call .predict()

That is the SAME pattern used for every other
scikit-learn model
(other regressions, classification, and beyond).
Learning it once carries over.

The model exposes its learned parameters after fitting:
- .coef_      the slope (an array, one value per feature)
- .intercept_ the intercept (a single number)

This is a Markdown cell.

In [ ]:
# Section 5 Python cell

LOG.info("Fitting a linear regression (scikit-learn)")

# Create the model object, then fit to the data.
model = LinearRegression()
model.fit(X, y)

# coef_ is an array (one slope per feature); we have one feature.
slope: float = float(model.coef_[0])
intercept: float = float(model.intercept_)
LOG.debug(f"  slope:     {slope:.6g}")
LOG.debug(f"  intercept: {intercept:.6g}")

LOG.info("Fitted line:")
LOG.info(f"  {TARGET_COL} = {slope:.6g} * {FEATURE_COL} + {intercept:.6g}")

# OPTIONAL sanity check.
# numpy can fit the same straight line: degree 1 returns
# [slope, intercept].  Check these values match.
np_slope, np_intercept = np.polyfit(X.ravel(), y, 1)
LOG.debug(f"  numpy check -> slope {np_slope:.6g}, intercept {np_intercept:.6g}")

## Section 6. Predict

WHY: The fitted values (y_hat) are what 
the line says y "should" be for each observed x. 

Compare them to the real y to see 
how far off the line is. 

Show a single prediction for a chosen
feature value as an example of using the model.

This is a Markdown cell.

In [ ]:
# Section 6 Python cell

LOG.info("Computing fitted values for every observed row")
y_hat: np.ndarray = model.predict(X)

LOG.info(f"Predicting {TARGET_LABEL} for one example {FEATURE_LABEL} value")

In [ ]:
# Section 6 Python cell

# The model expects a 2-D input of shape (n, 1), even for one value.
X_example: np.ndarray = np.array([[EXAMPLE_FEATURE_VALUE]])
y_example: float = float(model.predict(X_example)[0])

LOG.debug(f"  example {FEATURE_COL}: {EXAMPLE_FEATURE_VALUE:.6g}")
LOG.debug(f"  predicted {TARGET_COL}: {y_example:.6g}")

## Section 7. Examine the Fit (Residuals, R-squared, RMSE)

WHY: A line can be fit to ANY pair of numeric columns. 
 
These numbers are how we decide 
whether the line is a reasonable description
of the data:

- residual = actual y - fitted y_hat (one per row)
  How far each point sits above (+) or below (-) the line.

- R-squared: the fraction of variation in y the line accounts for.
  Ranges roughly 0 to 1. Higher means the line explains more.

- RMSE: root mean squared error, in the same units as y.
  The typical size of a residual.

The function computes and reports. 
It does NOT declare the fit "good" or "bad".
You read the numbers and the residual plot
to make a determination.

This is a Markdown cell.

In [ ]:
# Section 7 Python cell

LOG.info("Computing residuals (actual - fitted)")
residuals: np.ndarray = y - y_hat

# R-squared straight from the model
# (sklearn's .score is R-squared for regression).
# Equivalent to comparing the line to a flat mean line.
r_squared: float = float(model.score(X, y))

# RMSE: square residuals, average them, take square root.
rmse: float = float(np.sqrt(np.mean(residuals**2)))

LOG.info("Fit numbers (requires interpretation):")
LOG.debug(f"  R-squared: {r_squared:.4f}")
LOG.debug(f"  RMSE:      {rmse:.6g}  (in units of {TARGET_LABEL})")
LOG.debug(f"  residual min:  {float(np.min(residuals)):.6g}")
LOG.debug(f"  residual max:  {float(np.max(residuals)):.6g}")
LOG.debug(f"  residual mean: {float(np.mean(residuals)):.6g}")

LOG.info("""
CUSTOM: Update these notes and use Markdown cells to narrate what you see.

How to read these (this is YOUR judgment, not the script's):

- R-squared near 1: the line accounts for most of the variation in y.
- R-squared near 0: the line accounts for almost none.
- RMSE: the typical distance between a point and the line, in y's units.
- Residual plot:
  - if a straight line fits well, residuals
    scatter randomly around 0 with no pattern.
  - A curve,
  - a funnel (spread that grows or shrinks),
  - or clusters are signs a straight line
    is NOT the right description, which can be a useful finding.

There is no threshold that decides this for you.
Look at the numbers and the plots together and
write down what you conclude.
""")

CUSTOM: Update these notes and use Markdown cells to narrate and tell the story as you explore. For example:

Interpretation:

 - Values close to 1 (dark red) = strong positive correlation (both increase together)
 - Values close to -1 (dark blue) = strong negative correlation (one increases, other decreases)
 - Values close to 0 (white) = little or no linear relationship
 - The diagonal is always 1 (each variable correlates perfectly with itself)

From this heatmap, we can see that **gpp** and **co2** show strong positive correlation (~0.98).

This is a Markdown cell.

## Section 8. Make Plots

Create simple, notebook-friendly plots.

WHY: Visualizations reveal patterns not obvious in tables.
CUSTOM: Charts will vary depending on the dataset
        and questions of interest.

WHY: 
The fitted-line plot shows whether the line tracks the points.
The residual plot shows whether what's left over has a pattern.
Together they show whether a straight line is a fair description.

Common charts here:
1. A scatter of feature vs target with the fitted line drawn on top.
2. A residual plot: residuals vs the feature, with a line at zero.

This is a Markdown cell.

In [ ]:
# Section 8 Python cell - use "Run All" so prior cells are executed first.

feature_values: np.ndarray = df_model[FEATURE_COL].to_numpy()
target_values: np.ndarray = df_model[TARGET_COL].to_numpy()

LOG.info("---- Creating Scatter Plot with Fitted Line ----------")
LOG.info(f"----   Set x to {FEATURE_LABEL} -----------------------")
LOG.info(f"----   Set y to {TARGET_LABEL} -------------------------")

# Open a fresh blank canvas before a new chart
plt.figure()

# The observed points
scatter_plt: Axes = sns.scatterplot(
    x=feature_values,
    y=target_values,
)

# The fitted line. Sort by x so the line is drawn left to right.
order: np.ndarray = np.argsort(feature_values)
scatter_plt.plot(feature_values[order], y_hat[order])

scatter_plt.set_xlabel(FEATURE_LABEL)
scatter_plt.set_ylabel(TARGET_LABEL)
scatter_plt.set_title(f"{FEATURE_LABEL} vs {TARGET_LABEL} with fitted line")

# IN NOTEBOOK: SHOW AS YOU GO
#      plt.show() displays the current chart and closes it
#      Call this before starting a new chart
#      or next chart will be drawn on top of this one
# IN SCRIPT: WAIT TO SHOW TILL THE END
#      Do not call plt.show() here - let figures accumulate
#      so all charts display together with sequential Figure numbers.
#      plt.show() is called once at the end of make_plots()
plt.show()

In [ ]:
LOG.info("------ Creating Residual Plot --------------------------")
LOG.info(f"------   Set x to {FEATURE_LABEL} ----------------------")
LOG.info("------   Set y to the residual (actual - fitted) -------")

# Open a fresh blank canvas before a new chart
plt.figure()

residual_plt: Axes = sns.scatterplot(
    x=feature_values,
    y=residuals,
)

# A reference line at residual = 0. Points scattered randomly around
# this line (no pattern) is what a good straight-line fit looks like.
residual_plt.axhline(0)

residual_plt.set_xlabel(FEATURE_LABEL)
residual_plt.set_ylabel(f"Residual ({TARGET_LABEL})")
residual_plt.set_title(f"Residuals vs {FEATURE_LABEL}")

plt.show()

## Section 9. Summary and Next Steps

At the end, of your notebook, provide:

-  brief summary of your findings 
-  suggested next steps

WHY: Regression is not a final answer. 
The summary records what was fit 
and points to what the analyst still has to decide.

This is a Markdown cell.


In [ ]:
# Section 9 Python cell

LOG.info("========================")
LOG.info("SUMMARY")
LOG.info("========================")
LOG.info(f"Dataset: {DATASET_NAME}")
LOG.info(f"Feature (x): {FEATURE_COL}")
LOG.info(f"Target  (y): {TARGET_COL}")

LOG.info(f"Original rows: {df.shape[0]}")
LOG.info(f"Model rows:    {df_model.shape[0]}")

LOG.info("Fitted line:")
LOG.info(f"  {TARGET_COL} = {slope:.6g} * {FEATURE_COL} + {intercept:.6g}")

LOG.info("======================")
LOG.info("Review the fit numbers (R-squared, RMSE). ")
LOG.info("Look at the fitted-line plot and the residual plot. ")
LOG.info("Decide if a `straight line` is a fair description. ")
LOG.info("If the residuals DO show a pattern (e.g. curve, funnel, clusters),")
LOG.info("then a straight line is NOT a good description. ")
LOG.info("If the residuals DO NOT show a pattern,")
LOG.info("then a straight line MIGHT be a good description. ")
LOG.info("Either way, the findings may be valuable.")
LOG.info("======================")
LOG.info("Repeat with a different feature, or a transformed feature, ")
LOG.info("to investigate other options.")
LOG.info("======================")
LOG.info("Include instructions and specifics in your README.md file.")
LOG.info("Write up your narrative on your docs/index.md file.")
LOG.info("Include your next step suggestions for further analysis or modeling.")
LOG.info("======================")


LOG.info("EDA workflow complete")
LOG.info("========================")
LOG.info("Executed successfully!")
LOG.info("========================")

## Reminder: Run All before sending to GitHub

Before saving a notebook (and running git add-commit-push), click 'Run All' to generate all outputs and display them in the notebook. 

This is a Markdown cell.
